# Week 3 Portfolio Risk Engine

This notebook packages the reusable Week 3 portfolio APIs into one reproducible workflow. Financial calculations are delegated to `mrrp`; the notebook only prepares demo inputs and displays package outputs.

In [ ]:
import sys
from pathlib import Path

project_root = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

from mrrp.data.cache import load_parquet  # noqa: E402
from mrrp.portfolio import (  # noqa: E402
    build_correlation_table,
    build_exposure_table,
    build_portfolio_risk_summary,
    build_risk_contribution_table,
    build_summary_cards,
    compute_asset_returns,
    compute_portfolio_returns,
    load_asset_metadata,
    load_portfolio_config,
)
from mrrp.risk import (  # noqa: E402
    build_correlation_summary,
    classify_concentration_risk,
    compute_effective_num_holdings,
    compute_portfolio_beta,
    compute_top_n_weight,
)

## 1. Locate Cached ETF Prices

The normal path uses the processed parquet cache. A deterministic fallback is created after loading the portfolio configuration if the cache is absent or incomplete.

In [ ]:
data_path = project_root / "data/processed/adjusted_close.parquet"
cached_prices = load_parquet(data_path) if data_path.exists() else None
data_path

## 2. Load the Portfolio Configuration

The YAML file is the reproducible source for portfolio name, benchmark, currency, shorting policy, tickers, and weights.

In [ ]:
config = load_portfolio_config(project_root / "configs/sample_portfolio.yaml")
config

## 3. Compute Asset Returns

The cached demo uses recent complete observations. The fallback is deterministic, positive, and long enough for the 63-observation correlation regime.

In [ ]:
def deterministic_price_sample(tickers: list[str]) -> pd.DataFrame:
    index = pd.date_range("2024-01-02", periods=260, freq="B")
    steps = np.arange(len(index) - 1, dtype=float)
    sample: dict[str, np.ndarray] = {}
    for position, ticker in enumerate(tickers):
        periodic_returns = (
            0.0002
            + 0.00004 * position
            + 0.006 * np.sin(steps / 11 + position * 0.7)
            + 0.003 * np.cos(steps / 5 + position * 0.3)
        )
        sample[ticker] = np.concatenate(
            ([100.0], 100.0 * np.cumprod(1 + periodic_returns))
        )
    return pd.DataFrame(sample, index=index)


selected_tickers = list(dict.fromkeys([*config.holdings.index, config.benchmark]))
cache_is_usable = cached_prices is not None and set(selected_tickers).issubset(
    cached_prices.columns
)
if cache_is_usable:
    prices = cached_prices.loc[:, selected_tickers].dropna().tail(756)
    cache_is_usable = len(prices) >= 64
if not cache_is_usable:
    prices = deterministic_price_sample(selected_tickers)

data_source = "processed parquet cache" if cache_is_usable else "deterministic fallback"
asset_prices = prices.loc[:, config.holdings.index]
asset_returns = compute_asset_returns(asset_prices)
print(f"Data source: {data_source}; prices shape: {prices.shape}")
asset_returns.tail()

## 4. Compute Portfolio Returns

The return engine aligns columns to the validated weight order and does not silently normalize weights.

In [ ]:
portfolio_returns = compute_portfolio_returns(asset_returns, config.holdings)
portfolio_returns.rename("portfolio_return").tail().to_frame()

## 5. Concentration Risk

Effective holdings and top-three weight distinguish genuine breadth from a portfolio that only has many ticker symbols.

In [ ]:
concentration = pd.Series(
    {
        "concentration_label": classify_concentration_risk(config.holdings),
        "effective_holdings": compute_effective_num_holdings(config.holdings),
        "top_3_weight": compute_top_n_weight(config.holdings, n=3),
    },
    name="value",
)
concentration.to_frame()

## 6. Sector, Region, Style, and Asset-Class Proxy Exposure

Metadata is ETF-level proxy information. If the metadata YAML is unavailable, the deterministic mapping below keeps the offline demo runnable.

In [ ]:
metadata_path = project_root / "configs/asset_metadata.yaml"
fallback_metadata = {
    "SPY": {
        "asset_class": "Equity",
        "region": "United States",
        "style": "Large Blend",
        "sector_proxy": "Broad Market",
    },
    "QQQ": {
        "asset_class": "Equity",
        "region": "United States",
        "style": "Large Growth",
        "sector_proxy": "Technology",
    },
    "XIU.TO": {
        "asset_class": "Equity",
        "region": "Canada",
        "style": "Large Blend",
        "sector_proxy": "Broad Market",
    },
    "EFA": {
        "asset_class": "Equity",
        "region": "Developed ex North America",
        "style": "Large Blend",
        "sector_proxy": "Broad Market",
    },
    "EEM": {
        "asset_class": "Equity",
        "region": "Emerging Markets",
        "style": "Large Blend",
        "sector_proxy": "Broad Market",
    },
    "XLK": {
        "asset_class": "Equity",
        "region": "United States",
        "style": "Large Growth",
        "sector_proxy": "Technology",
    },
}
asset_metadata = (
    load_asset_metadata(metadata_path) if metadata_path.exists() else fallback_metadata
)
exposure_table = build_exposure_table(config, asset_metadata)
exposure_table

## 7. Correlation Regime

The matrix shows current full-sample relationships; the diagnostic row classifies the latest 63-observation mean pairwise correlation against its history.

In [ ]:
correlation_table = build_correlation_table(prices, config)
correlation_diagnostics = build_correlation_summary(
    asset_returns,
    config.holdings,
    window=63,
)
correlation_diagnostics

In [ ]:
correlation_table

## 8. Beta vs Benchmark

Portfolio beta estimates sensitivity to the configured benchmark after aligning return dates.

In [ ]:
benchmark_returns = compute_asset_returns(prices.loc[:, [config.benchmark]]).iloc[:, 0]
portfolio_beta = compute_portfolio_beta(portfolio_returns, benchmark_returns)
portfolio_beta

## 9. Risk Contribution

Component and percentage contributions show which configured holdings drive portfolio variance.

In [ ]:
risk_contribution_table = build_risk_contribution_table(prices, config)
risk_contribution_table

## 10. Full PortfolioRiskSummary

The typed summary and plain summary-card dictionary are the primary scalar interfaces for the Week 4 dashboard and later memo generation.

In [ ]:
risk_summary = build_portfolio_risk_summary(
    prices,
    config,
    asset_metadata=asset_metadata,
)
summary_cards = build_summary_cards(risk_summary)
pd.Series(summary_cards, name="value").to_frame()

## Interpretation and Limitations

- These metrics describe historical sample behavior; they are not forecasts.
- ETF metadata fields are broad proxies rather than holdings-level look-through analysis.
- Historical VaR/CVaR, covariance, correlation, and beta can change materially across samples.
- The deterministic fallback exists only for offline demonstration and testing.
- This research output is not investment advice.